# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sakhawat-4/my-ml-internship-v2/blob/main/work/notebooks/w04_signal_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

key_fields = ["impressions_90d", "clicks_90d", "ctr", "avg_position", "content_age_days", "engagement_rate"]
print(df[key_fields].describe(percentiles=[0.5, 0.9, 0.99]).T[["mean", "50%", "90%", "99%", "max"]])

print("\nHeavy-tail check (99th percentile vs max):")
for col in ["impressions_90d", "clicks_90d"]:
    p99 = df[col].quantile(0.99)
    print(f"  {col}: 99th pct = {p99:,.0f}, max = {df[col].max():,.0f} "
          f"({df[col].max()/max(p99,1):.1f}x the 99th percentile)")
print("-> impressions/clicks are heavily right-skewed, which is why the model features use log1p.")

                         mean     50%       90%        99%       max
impressions_90d   5200.366300  731.00  12136.40  73505.830  517715.0
clicks_90d          16.097333    1.00     32.00    253.010    4178.0
ctr                  0.510733    0.07      0.65      8.330     100.0
avg_position        16.342380   10.80     36.80     69.901     245.0
content_age_days   256.167800  236.00    463.00    537.000     564.0
engagement_rate      2.534520    0.00      6.94     33.330     100.0

Heavy-tail check (99th percentile vs max):
  impressions_90d: 99th pct = 73,506, max = 517,715 (7.0x the 99th percentile)
  clicks_90d: 99th pct = 253, max = 4,178 (16.5x the 99th percentile)
-> impressions/clicks are heavily right-skewed, which is why the model features use log1p.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [2]:
results = []

# --- Signal test 1: staleness (freshness_tier) -> decline rate ---
by_freshness = df.groupby("freshness_tier")["is_declining_label"].agg(["mean", "count"]).sort_values("mean", ascending=False)
print("Signal 1 — freshness_tier vs decline rate:")
print(by_freshness)
gap1 = by_freshness["mean"].max() - by_freshness["mean"].min()
verdict1 = "CONFIRMED" if gap1 > 0.05 else "MIXED"
print(f"Spread across tiers: {gap1:.3f} -> verdict: {verdict1}\n")
results.append(("staleness (freshness_tier)", verdict1, f"spread={gap1:.3f}"))

# --- Signal test 2: CTR-vs-position-tier -> decline rate ---
df["tier_median_ctr"] = df.groupby("position_tier")["ctr"].transform("median")
df["ctr_below_tier_median"] = df["ctr"] < df["tier_median_ctr"]
by_ctr = df.groupby("ctr_below_tier_median")["is_declining_label"].agg(["mean", "count"])
print("Signal 2 — below-tier-median CTR vs decline rate:")
print(by_ctr)
gap2 = by_ctr.loc[True, "mean"] - by_ctr.loc[False, "mean"]
verdict2 = "CONFIRMED" if gap2 > 0.03 else ("OPPOSITE" if gap2 < -0.03 else "MIXED")
print(f"Gap (below-median minus above-median): {gap2:.3f} -> verdict: {verdict2}\n")
results.append(("CTR vs position-tier median", verdict2, f"gap={gap2:.3f}"))

# --- Signal test 3: AI-referral share -> engagement rate ---
df["high_ai_traffic"] = df["ai_traffic_pct"] > df["ai_traffic_pct"].median()
by_ai = df.groupby("high_ai_traffic")["engagement_rate"].agg(["mean", "count"])
print("Signal 3 — above-median AI-referral share vs engagement rate:")
print(by_ai)
gap3 = by_ai.loc[True, "mean"] - by_ai.loc[False, "mean"]
verdict3 = "CONFIRMED" if gap3 > 0.02 else ("OPPOSITE" if gap3 < -0.02 else "MIXED")
print(f"Gap (high-AI minus low-AI engagement): {gap3:.3f} -> verdict: {verdict3}")
results.append(("AI-referral share vs engagement", verdict3, f"gap={gap3:.3f}"))

print("\n=== Summary ===")
for name, verdict, detail in results:
    print(f"  {name}: {verdict} ({detail})")

Signal 1 — freshness_tier vs decline rate:
                    mean  count
freshness_tier                 
91-180          0.611057   9171
31-90           0.588571    175
0-30            0.511377  20480
181+            0.471264    174
Spread across tiers: 0.140 -> verdict: CONFIRMED

Signal 2 — below-tier-median CTR vs decline rate:
                           mean  count
ctr_below_tier_median                 
False                  0.502630  16921
True                   0.593088  13079
Gap (below-median minus above-median): 0.090 -> verdict: CONFIRMED

Signal 3 — above-median AI-referral share vs engagement rate:
                     mean  count
high_ai_traffic                 
False            2.481690  28070
True             3.302886   1930
Gap (high-AI minus low-AI engagement): 0.821 -> verdict: CONFIRMED

=== Summary ===
  staleness (freshness_tier): CONFIRMED (spread=0.140)
  CTR vs position-tier median: CONFIRMED (gap=0.090)
  AI-referral share vs engagement: CONFIRMED (gap=0.821

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [3]:
# FlyRank's refresh flag assumes the "91-180" freshness tier is the highest-risk tier
# (this is the exact assumption the Week 4 baseline rule's "refresh" branch relies on).
# Test: is 91-180 actually the tier with the highest observed decline rate, or did the
# rule just pick a plausible-sounding tier?

flag_assumption_tier = "91-180"
rank_position = list(by_freshness.index).index(flag_assumption_tier) + 1
print(f"Flag assumes '{flag_assumption_tier}' is highest-risk.")
print(f"Observed rank of '{flag_assumption_tier}' by decline rate: #{rank_position} of {len(by_freshness)} tiers.")
print(by_freshness)

if rank_position == 1:
    flag_verdict = "CONFIRMED"
elif rank_position <= 2:
    flag_verdict = "MIXED"
else:
    flag_verdict = "OPPOSITE"
print(f"\nFlag-linked verdict: {flag_verdict}")

Flag assumes '91-180' is highest-risk.
Observed rank of '91-180' by decline rate: #1 of 4 tiers.
                    mean  count
freshness_tier                 
91-180          0.611057   9171
31-90           0.588571    175
0-30            0.511377  20480
181+            0.471264    174

Flag-linked verdict: CONFIRMED


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [4]:
print("What a content team should take from this audit:")
print(f"""
1. Staleness ({results[0][1]}) and the flag-linked freshness check ({flag_verdict}) both
   point the same direction: pages sitting in the mid-age band carry the most observed
   decline risk, which is a reasonable basis for a refresh flag -- but the effect size
   ({results[0][2]}) is directional, not dramatic, so it should rank pages rather than
   gate a single hard cutoff.
2. Signal 2 ({results[1][1]}) means CTR-below-tier-median is a real, usable secondary
   signal for a fix_ctr action -- it should stay in the score.
3. Signal 3 ({results[2][1]}) is exploratory only (AI-referral sessions are sparse per
   the lane guidance) and should not drive an action on its own.
""")

What a content team should take from this audit:

1. Staleness (CONFIRMED) and the flag-linked freshness check (CONFIRMED) both
   point the same direction: pages sitting in the mid-age band carry the most observed
   decline risk, which is a reasonable basis for a refresh flag -- but the effect size
   (spread=0.140) is directional, not dramatic, so it should rank pages rather than
   gate a single hard cutoff.
2. Signal 2 (CONFIRMED) means CTR-below-tier-median is a real, usable secondary
   signal for a fix_ctr action -- it should stay in the score.
3. Signal 3 (CONFIRMED) is exploratory only (AI-referral sessions are sparse per
   the lane guidance) and should not drive an action on its own.



## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.